In [ ]:
import sys, torch

if not torch.cuda.is_available():
    print("=" * 60)
    print("  ERROR: No GPU detected!")
    print("  This notebook requires a GPU runtime.")
    print("  Go to Runtime > Change runtime type > Hardware accelerator: GPU")
    print("=" * 60)
    sys.exit(1)

print(f"GPU OK: {torch.cuda.get_device_name(0)}  ({torch.cuda.device_count()} device(s))")
print(f"Torch CUDA: {torch.version.cuda}")

# Continuous Curriculum BC + PPO Training for Bomberland

This notebook trains a hybrid neural + codex agent using continuous curriculum RL.

Opponent difficulty ramps smoothly based on training progress, with
strength-weighted terminal rewards and adaptive priority adjustment.

---
**Workflow**:
1. Clone repo + install deps
2. Behavior Cloning (BC) pretraining
3. Single PPO training run with continuous opponent schedule
4. Export agent
5. Benchmark against codex agents
6. (Optional) Resume/finetune from best milestone

In [ ]:
# Cell 1: Clone repo, apply patches (if needed), install deps
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/BFCmath/AIC-GDGoC-2026.git"
REPO_DIR = Path("/content/AIC-GDGoC-2026")

if "google.colab" in sys.modules:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(str(REPO_DIR))
else:
    REPO_DIR = Path.cwd()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repo:", REPO_DIR)

# Apply RL training patches if not already applied.
# Sentinel: stage_train_bc_ppo.py is only created by the patch.
if not (REPO_DIR / "scripts" / "participant" / "stage_train_bc_ppo.py").exists():
    PATCHES_DIR = REPO_DIR / "docs" / "patches"
    if PATCHES_DIR.exists():
        p1 = PATCHES_DIR / "rl_training_patch (1).diff"
        if p1.exists():
            print(f"Applying {p1.name}...")
            subprocess.run(["patch", "-p1", "-i", str(p1)], cwd=REPO_DIR, check=True)
        p2 = PATCHES_DIR / "rl_stagewise_patch.diff"
        if p2.exists():
            print(f"Applying {p2.name}...")
            subprocess.run(["patch", "-p2", "-i", str(p2)], cwd=REPO_DIR, check=True)
        print("Patches applied.")
    else:
        print("ERROR: patches not found at docs/patches/. Cannot train.")
        sys.exit(1)
else:
    print("Repo already patched, skipping patch step.")

# Install deps filtering out torch/tf/sb3 (Colab has CUDA torch pre-installed)
reqs = Path("requirements.txt").read_text().splitlines()
reqs = [r for r in reqs if r and not r.startswith("#")
        and not r.startswith("torch")
        and not r.startswith("tensorflow")
        and not r.startswith("stable-baselines3")]
if reqs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + reqs, check=True)
print("Dependencies installed.")

def stream_run(cmd, check=True):
    """Run cmd with real-time stdout/stderr streaming."""
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return proc.returncode



## Configuration

The recommended config (profile `kaggle_medium`):
- BC: 80 matches, 35 epochs
- PPO: 900 total updates, 16 envs
- LR: 3e-4 → 6e-5 (smooth anneal)
- Entropy: 0.018 → 0.006
- Horizon: 128 → 256, MaxSteps: 160 → 500
- Opponents: schedule-based (8 agents, weak→strong)

Edit the cell below to customize.

In [ ]:
# Cell 2: Configuration — edit these values as needed
PROFILE = "kaggle_medium"       # smoke | kaggle_medium | kaggle_long
DEVICE = "cuda"                  # cuda or cpu
EVAL_MATCHES = 16                # matches per evaluation probe
TORCH_THREADS = 1                # 1 for Colab to avoid oversubscription
EXPORT_DIR = "exports/continuous_curriculum_agent"
SEED = 86
MILESTONE_DIR = "checkpoints/milestones"

# Resume from a previous run:
RESUME_CHECKPOINT = ""            # set to resume checkpoint path (e.g. "checkpoints/milestones/milestone_55pct_updateX.pt")
SKIP_BC = False                  # set True to skip BC (needed when resuming)

print(f"Profile: {PROFILE}")
print(f"Device: {DEVICE}")
print(f"Export dir: {EXPORT_DIR}")

In [ ]:
# Cell 2b: Override opponent schedule, reward config, hyperparams
# Edit the dict below, then run this cell before Cell 3.
# Each section is optional — omit sections you don't want to override.
from pathlib import Path
import json

CONFIG_OVERRIDES = {
    # Override opponent priority schedule:
    "opponent_schedule": [
        {"agent": "RandomAgent", "strength": 0.10, "start_pct": 0.00, "peak_end_pct": 0.08, "end_pct": 0.35, "priority": 8, "label": "random"},
        {"agent": "SimpleRuleAgent", "strength": 0.20, "start_pct": 0.00, "peak_end_pct": 0.12, "end_pct": 0.45, "priority": 8, "label": "simple_rule"},
        {"agent": "BoxFarmerAgent", "strength": 0.30, "start_pct": 0.08, "peak_end_pct": 0.22, "end_pct": 0.60, "priority": 8, "label": "box_farmer"},
        {"agent": "SmarterRuleAgent", "strength": 0.40, "start_pct": 0.12, "peak_end_pct": 0.30, "end_pct": 0.72, "priority": 8, "label": "smarter_rule"},
        {"agent": "TacticalRuleAgent", "strength": 0.50, "start_pct": 0.20, "peak_end_pct": 0.45, "end_pct": 0.90, "priority": 8, "label": "tactical_rule"},
        {"agent": "agent/codex/4.py", "strength": 0.70, "start_pct": 0.25, "peak_end_pct": 0.55, "end_pct": 1.00, "priority": 10, "label": "codex4"},
        {"agent": "agent/codex/8.py", "strength": 0.85, "start_pct": 0.35, "peak_end_pct": 0.65, "end_pct": 1.00, "priority": 10, "label": "codex8"},
        {"agent": "agent/codex/7.py", "strength": 1.00, "start_pct": 0.45, "peak_end_pct": 0.75, "end_pct": 1.00, "priority": 10, "label": "codex7"},
    ],
    # Override reward config:
    "reward": {
        "rank_rewards": [15.0, 0.0, -10.0, -10.0],
        "win_strength_offset": 0.5,
        "loss_penalty_mult": 0.75,
    },
    # Override hyperparams directly (overrides profile values):
    "hyperparam": {
        # "ppo_updates": 1200,
    },
}

path = Path("configs/overrides.json")
path.write_text(json.dumps(CONFIG_OVERRIDES, indent=2, default=str))
print(f"Wrote overrides to {path}")

## Main Training

Runs BC pretraining then a single continuous PPO run with smooth opponent
scheduling, hyperparameter annealing, and milestone checkpoints.

If `RESUME_CHECKPOINT` is set, resumes PPO from that checkpoint (with `SKIP_BC`).
Milestones saved to `checkpoints/milestones/`.

In [ ]:
# Cell 3: Run continuous curriculum BC + PPO training
train_flags = [
    sys.executable, "-u", "scripts/participant/stage_train_bc_ppo.py",
    "--profile", PROFILE,
    "--device", DEVICE,
    "--seed", str(SEED),
    "--torch-threads", str(TORCH_THREADS),
    "--export-dir", EXPORT_DIR,
    "--milestone-dir", MILESTONE_DIR,
]

if SKIP_BC:
    train_flags.append("--skip-bc")
if RESUME_CHECKPOINT:
    train_flags.extend(["--resume", RESUME_CHECKPOINT])
train_flags.extend(["--eval-matches", str(EVAL_MATCHES)])
if Path("configs/overrides.json").exists():
    train_flags.extend(["--overrides", "configs/overrides.json"])

print("Running:", " ".join(train_flags))
stream_run(train_flags)

## Export Agent

Exports the trained model + bundled codex agents into a standalone directory.
The exported `agent.py` uses a conservative neural override gate with teacher fallback.

In [ ]:
# Cell 4: Export the trained agent (prefer best.pt, fallback to final.pt)
checkpoint = Path(MILESTONE_DIR) / "best.pt"
if not checkpoint.exists():
    checkpoint = Path(MILESTONE_DIR) / "final.pt"

if checkpoint.exists():
    !python -u scripts/participant/train_bc_ppo.py \
        --mode export \
        --device cpu \
        --checkpoint $checkpoint \
        --export-dir $EXPORT_DIR
    print(f"Exported {checkpoint.name} to {EXPORT_DIR}")
else:
    print(f"No checkpoint found in {MILESTONE_DIR}.")

## Benchmark

Benchmark the exported agent against codex agents `4.py`, `7.py`, `8.py`.
Uses 50 matches, 400 max steps with 100ms timeout enforcement.

In [ ]:
# Cell 5: Benchmark exported agent
EXPORT_PATH = Path(EXPORT_DIR)
if EXPORT_PATH.exists():
    !python -m scripts.participant.benchmark \
        --agents $EXPORT_DIR agent/codex/4.py agent/codex/7.py agent/codex/8.py \
        --matches 50 \
        --max_steps 400 \
        --timeout \
        --timeout-ms 100
else:
    print(f"Export directory {EXPORT_DIR} not found.")

## Select Best Milestone Checkpoint

Do **not** assume the final checkpoint is best. Benchmark milestone checkpoints
and pick the one with the best win count, average rank, and timeout count.

The cell below exports and benchmarks each milestone checkpoint automatically.

In [ ]:
# Cell 6: Benchmark milestone checkpoints to select the best
from pathlib import Path
import subprocess, sys

CKPT_DIR = Path(MILESTONE_DIR)
if CKPT_DIR.exists():
    milestone_ckpts = sorted(CKPT_DIR.glob("milestone_*.pt"))
    for ckpt in milestone_ckpts:
        ckpt_name = ckpt.stem
        export_dir = f"exports/{ckpt_name}_agent"
        stream_run([
            sys.executable, "-u", "scripts/participant/train_bc_ppo.py",
            "--mode", "export", "--device", "cpu",
            "--checkpoint", str(ckpt), "--export-dir", export_dir,
        ])
        print(f"\n--- Benchmarking {ckpt_name} ---")
        stream_run([
            sys.executable, "-m", "scripts.participant.benchmark",
            "--agents", export_dir, "agent/codex/4.py", "agent/codex/7.py", "agent/codex/8.py",
            "--matches", "50", "--max_steps", "400", "--timeout", "--timeout-ms", "100",
        ])
else:
    print(f"No milestones found at {CKPT_DIR}.")

## (Optional) Resume / Finetune

If the main run completed but the agent is still weak, resume from the best
milestone checkpoint for additional PPO updates with a lower LR.
Set `RESUME_CKPT` and `ADDITIONAL_UPDATES` below.

In [ ]:
# Cell 7: (Optional) Resume finetune from a milestone checkpoint
RESUME_CKPT = "checkpoints/milestones/best.pt"
ADDITIONAL_UPDATES = 300

if Path(RESUME_CKPT).exists():
    !python -u scripts/participant/train_bc_ppo.py --mode ppo --device $DEVICE --checkpoint $RESUME_CKPT --save-checkpoint checkpoints/milestones/finetuned.pt --ppo-updates $ADDITIONAL_UPDATES --ppo-envs-per-update 16 --ppo-horizon-start 192 --ppo-horizon-end 256 --ppo-lr-start 5e-5 --ppo-lr-end 3e-5 --ppo-ent-start 0.008 --ppo-ent-end 0.004 --max-steps-start 400 --max-steps-end 500 --eval-interval 10 --eval-matches 20 --snapshot-interval 25 --torch-threads $TORCH_THREADS --overrides configs/overrides.json
    !python -u scripts/participant/train_bc_ppo.py --mode export --device cpu --checkpoint checkpoints/milestones/finetuned.pt --export-dir exports/finetuned_agent
    print("Finetune complete.")
else:
    print(f"{RESUME_CKPT} not found. Run the main training first.")

## Create Submission Zip

Package the best agent into a submission zip for Kaggle/Colab.

In [ ]:
# Cell 8: Create submission zip from exported agent
EXPORT_PATH = Path(EXPORT_DIR)
if EXPORT_PATH.exists():
    zip_name = EXPORT_PATH.name + "_submission.zip"
    !cd $EXPORT_DIR && zip -r ../../$zip_name agent.py model.pt 4.py 7.py 8.py
    print(f"Created {zip_name}")
else:
    print(f"Export directory {EXPORT_DIR} does not exist.")

## Notes

- Do **not** trust a checkpoint only because training reward improved. Select using benchmark + timeout.
- If neural-only/masked eval is weak, keep the conservative export gate (default). It behaves like a strong teacher-league agent with occasional learned overrides.
- If `fallback_rate` remains high through late training, increase BC matches/epochs before increasing PPO.
- The opponent schedule provides the curriculum. If a particular opponent dominates late training, adjust its `end_pct` or `priority` in overrides.
- Adaptive priority adjustment automatically tunes opponent difficulty: if win rate > 75%, priority decreases; if < 35%, priority increases.